# Harmonized Burning Activity Calendar — pipeline notebook

A thin driver over the `burncal` package (`src/burncal/`). All logic lives in the package and is covered by `pytest`; this notebook runs it for one region and draws the charts.

**Data source (`SOURCE`)**
- `"cube"`: real FIRMS data for Southeast Asia, from `scripts/build_cube.py` (`data/processed/cube.parquet`). Pick any country / province / Malaysian district.
- `"demo"`: synthetic data that checks the pipeline. ⚠️ **Never use demo results as findings.**

**Method:** NOAA-20 is the reference (Suomi NPP ended 2 Nov 2026). The other satellites are converted step by step: S-NPP → NOAA-21 → mean of the Aqua and Terra conversions (fitted 2012–2020, before the orbit drift; Aqua M3 + Terra M2, chosen by a backcast test) → Aqua alone (months without Terra) → Terra (Terra-only months before July 2002). The other steps choose a method with the paired one-standard-error rule.

## 1. Settings

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from burncal.config import Settings
from burncal.io import make_demo_data
from burncal.pipeline import run, run_cube, write_outputs, naive_series
from burncal.calendar import to_grid, anomaly_z, season_to_date, plot_calendar

CUBE_PATH = ROOT / "data" / "processed" / "cube.parquet"
SOURCE = "cube" if CUBE_PATH.exists() else "demo"

# Region (cube only). COUNTRY: ISO3 code, a list of codes, or None = all 11 Southeast Asian countries
# BRN Brunei, KHM Cambodia, IDN Indonesia, LAO Laos, MYS Malaysia, MMR Myanmar,
# PHL Philippines, SGP Singapore, THA Thailand, TLS Timor-Leste, VNM Vietnam
COUNTRY = "MYS"
ADM1 = None        # e.g. "Sarawak", "Riau", "Chiang Mai"; or a list
ADM2 = None        # Malaysian district, e.g. "Miri"

settings = Settings(country=COUNTRY, adm1=ADM1, adm2=ADM2)
print("source:", SOURCE)

## 2. Load and run

In [ ]:
if SOURCE == "cube":
    from burncal.cube import load_cube, platform_coverage
    cube = load_cube(CUBE_PATH)
    print("platform coverage:", platform_coverage(cube))
    res = run_cube(cube, settings)
    region = f"{COUNTRY or 'Southeast Asia'}" + (f" / {ADM1}" if ADM1 else "") + (f" / {ADM2}" if ADM2 else "")
else:
    res = run(make_demo_data(), Settings())
    region = "DEMO (synthetic)"
print("region:", region)
res.monthly.tail()

### Regions available in the cube

In [ ]:
if SOURCE == "cube":
    tbl = (cube[cube["country"].notna()].groupby(["country", "adm1"], dropna=False)["cell_days"].sum()
           .sort_values(ascending=False).rename("cell-days 2000-now"))
    display(tbl.groupby(level=0).sum().sort_values(ascending=False).to_frame())
    display(tbl.loc[COUNTRY].head(20).to_frame() if COUNTRY in tbl.index.get_level_values(0) else tbl.head(20).to_frame())

## 3. The problem: each platform sees a different amount of fire
Save this chart: it is the "before" picture.

In [ ]:
ts = res.monthly.index.to_timestamp()
fig, ax = plt.subplots(figsize=(12, 4))
for col in res.monthly.columns:
    ax.plot(ts, res.monthly[col], lw=.9, label=col)
ax.plot(ts, naive_series(res.monthly), color="black", ls="--", lw=1.4, alpha=.6, label="Naive record")
ax.set_yscale("symlog", linthresh=10); ax.set_ylabel(f"monthly {settings.unit}"); ax.legend(ncol=6)
ax.set_title(f"Raw monthly counts per platform — {region}"); plt.tight_layout(); plt.show()

## 4. Harmonization chain and benchmarks
For every step: the chosen method and **why**, leave-one-year-out scores, hold-out years, and the drift check (trend of the yearly mean log ratio; should be close to 0).

In [ ]:
for st in res.harmonization.steps:
    print(f"\n=== {st.donor}: {st.status} {st.reason}")
    if st.status != "applied":
        continue
    print(f"window {st.fit_window}, {st.n_pairs} months, filled {st.months_filled} months")
    print(f"method {st.method} — why: {st.selection}")
    print(f"drift {st.drift['slope_log_per_year']:+.3f} log/yr, band (log) {np.round(st.interval_log, 3)}")
    display(st.loyo.round(3))
    if st.holdout is not None and not st.holdout.empty:
        display(st.holdout.round(3))

## 5. Harmonized series, uncertainty and seam tests
`placebo_pct` = share of placebo steps (Januaries without a satellite change) smaller than the step at the seam. Values near 1 mean a seam is probably still there; El Niño years can also produce big steps, so read it with the chart.

In [ ]:
f = res.harmonization.frame
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts, naive_series(res.monthly), color="grey", lw=1, alpha=.6, label="Naive (with jump)")
ax.fill_between(ts, f["lower"], f["upper"], color="#d62728", alpha=.2, lw=0, label=f"{settings.interval:.0%} band")
ax.plot(ts, f["harmonized"], color="#d62728", lw=1.4, label=f"Harmonized ({settings.reference}-equivalent)")
for s in res.seams["seam"]:
    ax.axvline(pd.Period(s, "M").to_timestamp(), color="grey", ls=":")
ax.set_yscale("symlog", linthresh=10); ax.legend(); ax.set_title(f"After harmonization — {region}")
plt.tight_layout(); plt.show()
display(res.seams.round(3)); display(res.naive_seams.round(3))

## 6. Burning calendar
Counts (sequential) and anomaly (z-score of log counts vs the baseline, diverging). Dashed cells are converted estimates, not measured by NOAA-20. Year labels: **EN** = El Niño and **LN** = La Niña over the July–October fire season (NOAA ONI).

In [ ]:
try:
    from burncal.context import load_oni, enso_by_year
    enso = enso_by_year(load_oni())
    marks = {int(y): {"El Nino": "EN", "La Nina": "LN"}[l] for y, l in enso["label"].items() if l != "Neutral"}
except FileNotFoundError:
    marks = {}  # run scripts/download_context.py for El Nino labels

grid = to_grid(f["harmonized"])
estimated = to_grid(~f["source"].str.endswith("measured")).astype(bool)
plot_calendar(grid, f"Burning activity calendar — {region} — {settings.reference}-equivalent {settings.unit}",
              "YlOrRd", estimated=estimated, year_marks=marks, label=f"{settings.reference}-equivalent {settings.unit}")
plt.show()
z = anomaly_z(grid, settings.baseline_years)
plot_calendar(z, f"Anomaly vs {settings.baseline_years[0]}–{settings.baseline_years[1]} (z, log counts) — {region}",
              "RdBu_r", fmt=lambda v: f"{v:+.1f}", vmin=-3, vmax=3, estimated=estimated, year_marks=marks, label="z-score")
plt.show()

## 7. This season vs history (season-to-date)

In [ ]:
year = int(f["harmonized"].dropna().index.max().year)
std = season_to_date(f["harmonized"], year)
a = std.attrs
print(f"{region}, {year} Jan–month {a['through_month']}: rank {int(std.loc[year, 'rank'])} of "
      f"{std['to_date'].notna().sum()} years, {a['pct_vs_median']:+.0f}% vs median")
top = std.sort_values("rank").head(8)
top["enso"] = [marks.get(int(y), "") for y in top.index]
top

## 8. Exports (`outputs/`)

In [ ]:
for p in write_outputs(res, ROOT / "outputs"):
    print("wrote", p.relative_to(ROOT))

## Next steps
See `TODO.md`. The web app (`web/`, `npm run dev`) shows the same results interactively.